In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
llama_api_key = os.getenv("LLAMA_API_KEY")

file_name = "data_source\hipaa-simplification-201303.pdf"

<>:8: SyntaxWarning: invalid escape sequence '\h'
<>:8: SyntaxWarning: invalid escape sequence '\h'
C:\Users\alcid\AppData\Local\Temp\ipykernel_3012\1561652848.py:8: SyntaxWarning: invalid escape sequence '\h'
  file_name = "data_source\hipaa-simplification-201303.pdf"


In [2]:
from llama_parse import LlamaParse

parser = LlamaParse(
    api_key=llama_api_key,
    result_type="markdown",
    verbose=True,
    auto_mode=True
)

d:\School\CAP 5602\Project\RAG\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'validate_default' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'validate_default' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(


In [3]:
extra_info = {"file_name": file_name}

with open(f"{file_name}", "rb") as f:
    documents = parser.load_data(f, extra_info=extra_info)

with open("output.md", "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(doc.text)

Started parsing the file under job_id fa86bb86-77a7-407d-8be2-6853f66256f9


In [4]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(documents)

In [5]:
from llama_index.core.node_parser import SimpleNodeParser

node_parser = SimpleNodeParser()

In [6]:
from llama_index.llms.openai import OpenAI

query_engine = index.as_query_engine(
    llm=OpenAI(api_key=openai_api_key, temperature=0, model="gpt-5"))

In [7]:
user_query = "Explain HIPAA in simple terms. Send the response in json format with keys: 'summary', 'key_points'."
response = query_engine.query(user_query)
response

Response(response='{\n  "summary": "These rules explain your right to see and get copies of your health information held by a covered entity. You can choose the format (including electronic, when the records are kept electronically) if it’s readily producible. Access must be provided in a timely way, at a convenient time/place or by mail. Only limited, reasonable, cost-based fees may be charged. If access is denied, you must receive a clear written explanation, information on what you still can access, and, if applicable, how to ask for a review.",\n  "key_points": [\n    "You may inspect or get a copy (or both) of your protected health information in designated record sets.",\n    "If the same information exists in multiple places, it only needs to be produced once.",\n    "Form and format: you can get your information in the form/format you request if readily producible; otherwise, in a readable hard copy or another agreed format.",\n    "If records are kept electronically and you re

In [ ]:
import json

json_response = json.dumps(response.response, indent=4)

In [ ]:
import os
import json
import logging
from typing import Optional, List, Dict, Any
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

from llama_parse import LlamaParse
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
    Document
)
from llama_index.core.node_parser import (
    SentenceSplitter,
    SimpleNodeParser,
    HierarchicalNodeParser
)
from llama_index.core.extractors import (
    TitleExtractor,
    QuestionsAnsweredExtractor,
    KeywordExtractor
)
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore
from llama_index.core.storage.index_store import SimpleIndexStore
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.postprocessor import (
    SimilarityPostprocessor,
    KeywordNodePostprocessor
)
from llama_index.core.response.pprint_utils import pprint_response
from llama_index.core.schema import MetadataMode

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)


class EnhancedRAGSystem:

    def __init__(
        self,
        openai_api_key: Optional[str] = None,
        llama_api_key: Optional[str] = None,
        model: str = "gpt-5",
        embedding_model: str = "text-embedding-3-small",
        temperature: float = 0.1,
        chunk_size: int = 512,
        chunk_overlap: int = 50,
        storage_dir: str = "./storage",
        use_cache: bool = True
    ):

        if not openai_api_key or not llama_api_key:
            load_dotenv()
            self.openai_api_key = openai_api_key or os.getenv("OPENAI_API_KEY")
            self.llama_api_key = llama_api_key or os.getenv("LLAMA_API_KEY")
        else:
            self.openai_api_key = openai_api_key
            self.llama_api_key = llama_api_key

        if not self.openai_api_key or not self.llama_api_key:
            raise ValueError(
                "API keys must be provided or set in environment variables")

        self.model = model
        self.embedding_model = embedding_model
        self.temperature = temperature
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.storage_dir = Path(storage_dir)
        self.use_cache = use_cache

        self.storage_dir.mkdir(exist_ok=True)

        self._setup_models()

        self._setup_parser()

        self.index = None
        self.documents = []

        logger.info("Enhanced RAG System initialized successfully")

    def _setup_models(self):
        try:
            self.llm = OpenAI(
                api_key=self.openai_api_key,
                model=self.model,
                temperature=self.temperature,
                max_tokens=2048
            )

            self.embed_model = OpenAIEmbedding(
                api_key=self.openai_api_key,
                model=self.embedding_model
            )

            Settings.llm = self.llm
            Settings.embed_model = self.embed_model
            Settings.chunk_size = self.chunk_size
            Settings.chunk_overlap = self.chunk_overlap

            logger.info(
                f"Models configured: LLM={self.model}, Embeddings={self.embedding_model}")

        except Exception as e:
            logger.error(f"Error setting up models: {str(e)}")
            raise

    def _setup_parser(self):
        try:
            self.parser = LlamaParse(
                api_key=self.llama_api_key,
                result_type="markdown",
                verbose=True,
                auto_mode=True,
                language="en",
                parsing_instruction="Extract all text content preserving structure and formatting"
            )
            logger.info("LlamaParse initialized")
        except Exception as e:
            logger.error(f"Error setting up parser: {str(e)}")
            raise

    def parse_document(
        self,
        file_path: str,
        save_parsed: bool = True,
        output_dir: str = "./parsed_documents"
    ) -> List[Document]:

        try:
            file_path = Path(file_path)
            if not file_path.exists():
                raise FileNotFoundError(f"File not found: {file_path}")

            logger.info(f"Parsing document: {file_path}")

            extra_info = {
                "file_name": str(file_path),
                "file_type": file_path.suffix,
                "parsed_date": datetime.now().isoformat()
            }

            with open(file_path, "rb") as f:
                documents = self.parser.load_data(f, extra_info=extra_info)

            if save_parsed:
                output_path = Path(output_dir)
                output_path.mkdir(exist_ok=True)
                output_file = output_path / f"{file_path.stem}_parsed.md"

                with open(output_file, "w", encoding="utf-8") as f:
                    for doc in documents:
                        f.write(
                            f"# Document: {doc.metadata.get('file_name', 'Unknown')}\n\n")
                        f.write(doc.text)
                        f.write("\n\n---\n\n")

                logger.info(f"Parsed document saved to: {output_file}")

            self.documents.extend(documents)
            logger.info(f"Successfully parsed {len(documents)} document(s)")

            return documents

        except Exception as e:
            logger.error(f"Error parsing document: {str(e)}")
            raise

    def create_index(
        self,
        documents: Optional[List[Document]] = None,
        use_metadata_extractor: bool = True,
        persist: bool = True
    ) -> VectorStoreIndex:

        try:
            if self.use_cache and self._index_exists():
                logger.info("Loading existing index from storage")
                self.index = self._load_index()
                return self.index

            docs_to_index = documents or self.documents
            if not docs_to_index:
                raise ValueError("No documents to index")

            logger.info(
                f"Creating index from {len(docs_to_index)} document(s)")

            transformations = []

            node_parser = SentenceSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap,
                paragraph_separator="\n\n",
                secondary_chunking_regex="[.!?]"
            )
            transformations.append(node_parser)

            if use_metadata_extractor:
                transformations.extend([
                    TitleExtractor(llm=self.llm, num_workers=1),
                    KeywordExtractor(llm=self.llm, keywords=10),
                    QuestionsAnsweredExtractor(llm=self.llm, questions=3)
                ])

            pipeline = IngestionPipeline(transformations=transformations)

            nodes = pipeline.run(documents=docs_to_index)

            self.index = VectorStoreIndex(nodes=nodes)

            if persist:
                self._persist_index()

            logger.info(f"Index created with {len(nodes)} nodes")
            return self.index

        except Exception as e:
            logger.error(f"Error creating index: {str(e)}")
            raise

    def query(
        self,
        query_text: str,
        response_mode: str = "compact",
        similarity_top_k: int = 5,
        format_json: bool = False,
        json_keys: Optional[List[str]] = None
    ) -> Dict[str, Any]:

        try:
            if not self.index:
                raise ValueError(
                    "Index not created. Call create_index() first")

            logger.info(f"Processing query: {query_text[:100]}...")

            query_engine = self.index.as_query_engine(
                llm=self.llm,
                response_mode=response_mode,
                similarity_top_k=similarity_top_k,
                node_postprocessors=[
                    SimilarityPostprocessor(similarity_cutoff=0.3),
                    KeywordNodePostprocessor(
                        required_keywords=[],
                        exclude_keywords=[]
                    )
                ],
                verbose=True
            )

            if format_json and json_keys:
                json_instruction = f"\n\nFormat your response as JSON with these keys: {json_keys}"
                query_text = query_text + json_instruction

            response = query_engine.query(query_text)

            result = {
                "query": query_text,
                "response": str(response),
                "source_nodes": []
            }

            for node in response.source_nodes:
                source_info = {
                    "text": node.text[:200] + "...",
                    "score": node.score,
                    "metadata": node.metadata
                }
                result["source_nodes"].append(source_info)

            if format_json:
                try:
                    result["parsed_response"] = json.loads(str(response))
                except json.JSONDecodeError:
                    logger.warning("Could not parse response as JSON")

            logger.info("Query processed successfully")
            return result

        except Exception as e:
            logger.error(f"Error processing query: {str(e)}")
            raise

    def batch_query(
        self,
        queries: List[str],
        **query_kwargs
    ) -> List[Dict[str, Any]]:

        results = []
        for i, query in enumerate(queries, 1):
            logger.info(f"Processing query {i}/{len(queries)}")
            try:
                result = self.query(query, **query_kwargs)
                results.append(result)
            except Exception as e:
                logger.error(f"Error in query {i}: {str(e)}")
                results.append({"query": query, "error": str(e)})

        return results

    def update_index(
        self,
        new_documents: List[Document]
    ):

        try:
            if not self.index:
                raise ValueError(
                    "Index not created. Call create_index() first")

            logger.info(
                f"Updating index with {len(new_documents)} new document(s)")

            node_parser = SentenceSplitter(
                chunk_size=self.chunk_size,
                chunk_overlap=self.chunk_overlap
            )

            new_nodes = node_parser.get_nodes_from_documents(new_documents)

            for node in new_nodes:
                self.index.insert(node)

            self.documents.extend(new_documents)

            if self.use_cache:
                self._persist_index()

            logger.info("Index updated successfully")

        except Exception as e:
            logger.error(f"Error updating index: {str(e)}")
            raise

    def _persist_index(self):
        try:
            self.index.storage_context.persist(
                persist_dir=str(self.storage_dir))
            logger.info(f"Index persisted to {self.storage_dir}")
        except Exception as e:
            logger.error(f"Error persisting index: {str(e)}")
            raise

    def _load_index(self) -> VectorStoreIndex:
        try:
            from llama_index.core import load_index_from_storage
            storage_context = StorageContext.from_defaults(
                persist_dir=str(self.storage_dir)
            )
            index = load_index_from_storage(storage_context)
            logger.info(f"Index loaded from {self.storage_dir}")
            return index
        except Exception as e:
            logger.error(f"Error loading index: {str(e)}")
            raise

    def _index_exists(self) -> bool:
        index_file = self.storage_dir / "docstore.json"
        return index_file.exists()

    def clear_cache(self):
        try:
            import shutil
            if self.storage_dir.exists():
                shutil.rmtree(self.storage_dir)
                self.storage_dir.mkdir(exist_ok=True)
                logger.info("Cache cleared")
        except Exception as e:
            logger.error(f"Error clearing cache: {str(e)}")
            raise


def main():

    rag = EnhancedRAGSystem(
        model="gpt-5",  # or "gpt-4o" if available
        chunk_size=512,
        chunk_overlap=50,
        temperature=0.1,
        use_cache=True
    )

    file_path = "data_source/hipaa-simplification-201303.pdf"
    documents = rag.parse_document(
        file_path=file_path,
        save_parsed=True
    )

    index = rag.create_index(
        documents=documents,
        use_metadata_extractor=True,
        persist=True
    )

    result = rag.query(
        query_text="Explain HIPAA in simple terms",
        response_mode="compact",
        similarity_top_k=5,
        format_json=True,
        json_keys=["summary", "key_points", "implications"]
    )

    print("\n--- Query Result ---")
    print(f"Response: {result['response']}")
    print(f"\nSources used: {len(result['source_nodes'])}")

    queries = [
        "What are the main privacy requirements under HIPAA?",
        "How does HIPAA protect patient information?",
        "What are the penalties for HIPAA violations?"
    ]

    batch_results = rag.batch_query(
        queries=queries,
        response_mode="compact",
        similarity_top_k=3
    )

    print("\n--- Batch Query Results ---")
    for i, result in enumerate(batch_results, 1):
        print(f"\nQuery {i}: {result['query'][:50]}...")
        if 'error' in result:
            print(f"Error: {result['error']}")
        else:
            print(f"Response: {result['response'][:200]}...")


if __name__ == "__main__":
    main()